Hey there! 

This is the code we used for preprocessing our photovoice data from raw .json to geojson ready to be used for mapping. We additionally combined our data with participant data collected through a survey. I have removed all personal data before sharing, but kept some details from our dataset for you to see how the code works. 

_**Created by:** Csilla Duray & Sofie Burgos-Thorsen_

# 1. prepare environment

In [1]:
# pip install --use-pep517

In [2]:
# pip install google-colab

## 1.1 code for Google Colab (can be skipped) 

In [3]:
# import os
# from google.colab import drive

In [4]:
# drive.mount('/content/drive', force_remount=True)
# path = ''     # add your path here!
# os.chdir(path)

## 1.2 using another path (can be skipped)

In [5]:
'''
import os
path = ''       # add your path here!
os.chdir(path)
'''

"\nimport os\npath = ''       # add your path here!\nos.chdir(path)\n"

## 1.3 loading libraries

In [7]:
import json
import pandas as pd
from datetime import datetime

## 1.4 loading datasets

In [8]:
# annotation data (processed in points 2 and 3)
# new dataset downloaded!
with open('data-1720694198255.json') as f:
    raw_data = json.load(f)

In [9]:
# survey data (processed in point 4)
survey_df = pd.read_excel('LIVING LAB NORDHAVN survey (svar).xlsx')

In [10]:
# app data used for merging (point 5)
username_to_email = pd.read_excel('username_to_email.xlsx', header=None, names=['username', 'email'])

# 2. annotations: load to df

### 2.1 investigate structure (can be skipped)

In [ ]:
date = raw_data['photos'][0]['createdAt']
formatted_date = datetime.fromisoformat(date.replace("Z", "+00:00")).strftime("%d-%m-%Y")
formatted_date

'25-04-2024'

In [ ]:
raw_data["photos"][0]['id']

'76d4c873-3b4f-4fb9-9085-10b7104c4c1d'

In [ ]:
raw_data["photos"][0]['latitude']

55.71664

In [ ]:
raw_data["photos"][0]['longitude']

12.609841

In [ ]:
#fetching URLs for photos size 400 and size 1600
raw_data["photos"][0]["thumbnails"][1]['url']
raw_data["photos"][0]["thumbnails"][4]['url']

'https://urbanbelonging.app/minio/photo-uploads-prod/6160f900-5d6f-4cb0-8d23-39ecb1908205-1600.jpeg'

In [ ]:
raw_data["photos"][0]['annotationAnswers']

[{'answerType': 'slider',
  'sliderAnswer': 5,
  'textInputAnswer': None,
  'singleChoiceAnswer': None,
  'multipleChoiceAnswer': []},
 {'answerType': 'slider',
  'sliderAnswer': 5,
  'textInputAnswer': None,
  'singleChoiceAnswer': None,
  'multipleChoiceAnswer': []},
 {'answerType': 'multiple-choice',
  'sliderAnswer': None,
  'textInputAnswer': None,
  'singleChoiceAnswer': None,
  'multipleChoiceAnswer': ['Grass',
   'Tres',
   'Flowers',
   'Other vegetation',
   'Birds',
   'Sun']},
 {'answerType': 'multiple-choice',
  'sliderAnswer': None,
  'textInputAnswer': None,
  'singleChoiceAnswer': None,
  'multipleChoiceAnswer': ['Pretty', 'Wild']},
 {'answerType': 'multiple-choice',
  'sliderAnswer': None,
  'textInputAnswer': None,
  'singleChoiceAnswer': None,
  'multipleChoiceAnswer': ['Not accessible',
   'Peaceful',
   'Feels safe',
   'Feeling of belonging']},
 {'answerType': 'multiple-choice',
  'sliderAnswer': None,
  'textInputAnswer': None,
  'singleChoiceAnswer': None,
  'mu

In [ ]:
raw_data["photos"][0]['annotationAnswers'][0]['sliderAnswer']
raw_data["photos"][0]['annotationAnswers'][1]['sliderAnswer']
raw_data["photos"][0]['annotationAnswers'][2]['multipleChoiceAnswer']
raw_data["photos"][0]['annotationAnswers'][3]['multipleChoiceAnswer']
raw_data["photos"][0]['annotationAnswers'][4]['multipleChoiceAnswer']
raw_data["photos"][0]['annotationAnswers'][5]['multipleChoiceAnswer']
raw_data["photos"][0]['annotationAnswers'][1]['sliderAnswer']

5

In [ ]:
len(raw_data['photos'])

1757

### 2.2 create dataframe

In [ ]:
data = []
for p in range(len(raw_data['photos'])):
    temp = []
    for q in ['id', 'createdById', 'createdAt', 'latitude', 'longitude','thumbnails', 'annotationAnswers',]:
        if q == 'annotationAnswers':
            try:
                temp.append(raw_data['photos'][p][q][0]['sliderAnswer'])
            except:
                temp.append(None)
            try:
                temp.append(raw_data['photos'][p][q][1]['sliderAnswer'])
            except:
                temp.append(None)
            try:
                temp.append(raw_data['photos'][p][q][2]['multipleChoiceAnswer'])
            except:
                temp.append(None)
            try:
                temp.append(raw_data['photos'][p][q][3]['multipleChoiceAnswer'])
            except:
                temp.append(None)
            try:
                temp.append(raw_data['photos'][p][q][4]['multipleChoiceAnswer'])
            except:
                temp.append(None)
            try:
                temp.append(raw_data['photos'][p][q][5]['multipleChoiceAnswer'])
            except:
                temp.append(None)        
        elif q == 'createdAt':
          date = raw_data['photos'][p][q]
          formatted_date = datetime.fromisoformat(date.replace("Z", "+00:00")).strftime("%d-%m-%Y")
          formatted_datetime = datetime.fromisoformat(date.replace("Z", "+00:00")).strftime("%d-%m-%Y %H:%M:%S")
          temp.append(formatted_date)
          temp.append(formatted_datetime)
        elif q == 'thumbnails':
            # Check if 'thumbnails' key exists and has enough elements
            if q in raw_data['photos'][p] and len(raw_data['photos'][p][q]) > 1:
                url_thumbnail = raw_data['photos'][p][q][1].get('url', 'No URL')
                formatted_url_t = str(url_thumbnail)
                url_original = raw_data['photos'][p][q][4].get('url', 'No URL')
                formatted_url_o = str(url_original)
                temp.append(formatted_url_t)
                temp.append(formatted_url_o)
            else:
                temp.append('No URL')
        else:
            temp.append(raw_data['photos'][p][q])
    data.append(temp)

In [ ]:
len(data)

1757

In [ ]:
df = pd.DataFrame(data, columns = ['photo_id', 'user_id', 'date', 'datetime', 'lat', 'long', 'url_thumb','url_original','appeal',
                                   'biodiversity_perceived', 'nature_elements_lst',
                                   'nature_quality_lst', 'social_quality_lst', 'sounds_lst'])
print(len(df))
df.head(2)

1757


,photo_id,user_id,date,datetime,lat,long,url_thumb,url_original,appeal,biodiversity_perceived,nature_elements_lst,nature_quality_lst,social_quality_lst,sounds_lst
0,76d4c873-3b4f-4fb9-9085-10b7104c4c1d,f48014f1-41e6-490f-9fce-b769514e2c0f,25-04-2024,25-04-2024 09:25:38,55.71664,12.609841,https://urbanbelonging.app/minio/photo-uploads...,https://urbanbelonging.app/minio/photo-uploads...,5.0,5.0,"[Grass, Tres, Flowers, Other vegetation, Birds...","[Pretty, Wild]","[Not accessible, Peaceful, Feels safe, Feeling...","[Construction, Birds, Wind/weather, Traffic]"
1,da370966-93f3-4bfb-9ad1-b54aace7f320,98981014-9d6c-4e37-8394-494d4f0436a6,21-05-2024,21-05-2024 12:26:18,55.72324,12.608316,https://urbanbelonging.app/minio/photo-uploads...,https://urbanbelonging.app/minio/photo-uploads...,4.0,4.0,"[Tres, Grass, Flowers, Other vegetation, Soil,...","[Abundant, Messy, Wild]",[Peaceful],"[Construction, Birds, Wind/weather]"


In [ ]:
# add site data

walks_dict = {'Århusgadekvarteret': [['2024-04-15 7:00:00', '2024-04-15 8:00:00'], ['2024-04-23 10:00:00', '2024-04-23 11:00:00'], 
                                     ['2024-04-25 11:00:00', '2024-04-25 12:00:00']],
              'Orientkaj': [['2024-05-15 11:00:00', '2024-05-15 12:00:00'], ['2024-05-16 11:00:00', '2024-05-16 12:00:00'],
                            ['2024-05-21 11:00:00', '2024-05-21 12:00:00']],
              'Skudehavnen': [['2024-04-15 9:00:00', '2024-04-15 10:00:00'], ['2024-04-23 11:00:00', '2024-04-23 11:30:00'], 
                              ['2024-05-16 13:00:00', '2024-05-16 14:00:00']],
              'Nordhavnstippen': [['2024-04-15 8:00:00', '2024-04-15 9:00:00'], ['2024-04-23 11:30:00', '2024-04-23 12:30:00'],
                                  ['2024-04-25 9:40:00', '2024-04-25 10:20:00'], ['2024-05-15 12:00:00', '2024-05-15 13:00:00'], 
                                  ['2024-05-16 12:00:00', '2024-05-16 13:00:00'], ['2024-05-21 12:00:00', '2024-05-21 14:00:00']],
              'Tunnelfabrikken': [['2024-04-25 9:00:00', '2024-04-25 9:40:00'], ['2024-04-25 10:20:00', '2024-04-25 10:35:00'],
                                  ['2024-05-15 13:00:00', '2024-05-15 14:00:00']],
              'tbd': [['2024-04-25 7:30:00', '2024-04-25 7:45:00'], ['2024-05-13 7:30:00', '2024-05-13 10:45:00']]}

df['datetime'] = pd.to_datetime(df['datetime'])
df['site'] = None

for site, periods in walks_dict.items():
    for start_str, end_str in periods:
        start_time = pd.Timestamp(start_str)
        end_time = pd.Timestamp(end_str)
        df.loc[(df['datetime'] >= start_time) & (df['datetime'] <= end_time), 'site'] = site

In [ ]:
# add Fiskerihavnen (wasn't walked in a separate time)
lat_range = [55.725533, 55.722983]
long_range = [12.60228, 12.609673]

df.loc[(df['lat'] >= min(lat_range)) & (df['lat'] <= max(lat_range)) & 
       (df['long'] >= min(long_range)) & (df['long'] <= max(long_range)), 'site'] = 'Fiskerihavnen'

photo_ids = ['a88229c2-6cb9-4710-a165-3ffd9b625663', 'b0c8feb3-2965-4a38-b55c-1a94a26569c7',
             'b178cae0-14f6-4255-b7dd-4d93179733d4', '01815558-5823-4439-8dcb-d010d824b099',
             '2269f880-57f7-487f-ab32-2016320550bb', '7f332df7-eeeb-4b9d-8ea0-e8b9e37dba56']

df.loc[df['photo_id'].isin(photo_ids), 'site'] = 'Fiskerihavnen'

In [ ]:
# change format of numerical annotations
for col in 'appeal', 'biodiversity_perceived':
    df[col] = df[col].astype('Int64')

In [ ]:
# new col for filename
df['filename'] = df['url_original']
df['filename'] = df['filename'].str.replace('https://urbanbelonging.app/minio/photo-uploads-prod/', '', regex=False)

In [ ]:
df.head(2)

,photo_id,user_id,date,datetime,lat,long,url_thumb,url_original,appeal,biodiversity_perceived,nature_elements_lst,nature_quality_lst,social_quality_lst,sounds_lst,site,filename
0,76d4c873-3b4f-4fb9-9085-10b7104c4c1d,f48014f1-41e6-490f-9fce-b769514e2c0f,25-04-2024,2024-04-25 09:25:38,55.71664,12.609841,https://urbanbelonging.app/minio/photo-uploads...,https://urbanbelonging.app/minio/photo-uploads...,5,5,"[Grass, Tres, Flowers, Other vegetation, Birds...","[Pretty, Wild]","[Not accessible, Peaceful, Feels safe, Feeling...","[Construction, Birds, Wind/weather, Traffic]",Tunnelfabrikken,6160f900-5d6f-4cb0-8d23-39ecb1908205-1600.jpeg
1,da370966-93f3-4bfb-9ad1-b54aace7f320,98981014-9d6c-4e37-8394-494d4f0436a6,21-05-2024,2024-05-21 12:26:18,55.72324,12.608316,https://urbanbelonging.app/minio/photo-uploads...,https://urbanbelonging.app/minio/photo-uploads...,4,4,"[Tres, Grass, Flowers, Other vegetation, Soil,...","[Abundant, Messy, Wild]",[Peaceful],"[Construction, Birds, Wind/weather]",Fiskerihavnen,b66b51be-064e-4203-bc86-e6f81dd227c7-1600.jpeg


### 2.3 debugging: empty rows
- count: 296

In [ ]:
df[df.isna().T.any()]

,photo_id,user_id,date,datetime,lat,long,url_thumb,url_original,appeal,biodiversity_perceived,nature_elements_lst,nature_quality_lst,social_quality_lst,sounds_lst,site,filename
4,afa83188-3d10-4a92-8b38-82d919cb1df9,11e86d0a-649d-4cf5-ae4e-bbdee4343e30,15-04-2024,2024-04-15 07:12:22,55.706505,12.596408,https://urbanbelonging.app/minio/photo-uploads...,https://urbanbelonging.app/minio/photo-uploads...,<NA>,<NA>,None,None,None,None,Århusgadekvarteret,ea113a72-31b3-41b7-ab32-18f2574b06ce-1600.jpeg
5,defc78d4-4c78-46c0-acc5-ae1f7138b23c,932d2b73-1c8d-4709-a7fc-e2064b409693,25-04-2024,2024-04-25 09:53:02,55.719480,12.615249,https://urbanbelonging.app/minio/photo-uploads...,https://urbanbelonging.app/minio/photo-uploads...,<NA>,<NA>,None,None,None,None,Nordhavnstippen,05b4e74b-d41e-45ce-9af1-956ff7c65028-1600.jpeg
8,ef16040d-4b99-47f7-a1dd-f642d121a7f8,11e86d0a-649d-4cf5-ae4e-bbdee4343e30,15-04-2024,2024-04-15 07:12:38,55.706482,12.596223,https://urbanbelonging.app/minio/photo-uploads...,https://urbanbelonging.app/minio/photo-uploads...,<NA>,<NA>,None,None,None,None,Århusgadekvarteret,9da568a6-028c-4853-91ee-62b95facf555-1600.jpeg
13,3202c7db-3ca4-49ef-a7b4-0fa68348cedb,11e86d0a-649d-4cf5-ae4e-bbdee4343e30,15-04-2024,2024-04-15 07:13:29,55.706430,12.595624,https://urbanbelonging.app/minio/photo-uploads...,https://urbanbelonging.app/minio/photo-uploads...,<NA>,<NA>,None,None,None,None,Århusgadekvarteret,3c34b2e5-154d-4c07-b83f-112b93646733-1600.jpeg
17,4a7f052c-2114-45a0-9e25-02e38718b889,11e86d0a-649d-4cf5-ae4e-bbdee4343e30,15-04-2024,2024-04-15 07:14:12,55.706356,12.595533,https://urbanbelonging.app/minio/photo-uploads...,https://urbanbelonging.app/minio/photo-uploads...,<NA>,<NA>,None,None,None,None,Århusgadekvarteret,d0e02109-ffa8-468a-8250-40ad30c95ca2-1600.jpeg
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1612,ada18d7d-b08e-45f7-8652-3414880cb5bc,3b05ff7a-3d12-4bcd-9293-fd18b943de1b,21-05-2024,2024-05-21 11:14:54,55.711930,12.595744,https://urbanbelonging.app/minio/photo-uploads...,https://urbanbelonging.app/minio/photo-uploads...,<NA>,<NA>,None,None,None,None,Orientkaj,b7b845b4-1193-4b40-872a-9df35ea4a73f-1600.jpeg
1704,865982dd-91b8-492b-91bb-7ea1623bbada,d03defb6-aadc-4062-8eb3-7a1698133444,21-05-2024,2024-05-21 11:40:55,55.712000,12.595181,https://urbanbelonging.app/minio/photo-uploads...,https://urbanbelonging.app/minio/photo-uploads...,<NA>,<NA>,None,None,None,None,Orientkaj,0e511ee4-2663-4ba7-b2ec-7d04b6f5a310-1600.jpeg
1705,96c5b132-be0d-4bcd-a000-8ab68b853fca,90489203-f1da-4eb1-8c54-62e489cc5ed0,21-05-2024,2024-05-21 11:40:56,NaN,NaN,https://urbanbelonging.app/minio/photo-uploads...,https://urbanbelonging.app/minio/photo-uploads...,4,2,"[Grass, Stones/rocks, Water]",[Politically driven],"[Accessible, Stressful]","[Traffic, Construction, Water, Wind/weather]",Orientkaj,cc34e19e-6e05-4e97-b34f-be421e3048c0-1600.jpeg
1708,1d4341f4-98fb-4ed9-bf3d-884a0b6e284a,90489203-f1da-4eb1-8c54-62e489cc5ed0,21-05-2024,2024-05-21 11:40:39,NaN,NaN,https://urbanbelonging.app/minio/photo-uploads...,https://urbanbelonging.app/minio/photo-uploads...,<NA>,<NA>,None,None,None,None,Orientkaj,956df2fd-4ab0-4e46-96be-1a2d91cf6e49-1600.jpeg


# 3. annotations: preprocess and one-hot encode categorical data

In [ ]:
df['nature_elements_lst'] = df['nature_elements_lst'].fillna('').apply(list)
df['nature_elements_lst'] = df['nature_elements_lst'].apply(lambda x: ['Trees' if item == 'Tres' else item for item in x])
df.head(2)

,photo_id,user_id,date,datetime,lat,long,url_thumb,url_original,appeal,biodiversity_perceived,nature_elements_lst,nature_quality_lst,social_quality_lst,sounds_lst,site,filename
0,76d4c873-3b4f-4fb9-9085-10b7104c4c1d,f48014f1-41e6-490f-9fce-b769514e2c0f,25-04-2024,2024-04-25 09:25:38,55.71664,12.609841,https://urbanbelonging.app/minio/photo-uploads...,https://urbanbelonging.app/minio/photo-uploads...,5,5,"[Grass, Trees, Flowers, Other vegetation, Bird...","[Pretty, Wild]","[Not accessible, Peaceful, Feels safe, Feeling...","[Construction, Birds, Wind/weather, Traffic]",Tunnelfabrikken,6160f900-5d6f-4cb0-8d23-39ecb1908205-1600.jpeg
1,da370966-93f3-4bfb-9ad1-b54aace7f320,98981014-9d6c-4e37-8394-494d4f0436a6,21-05-2024,2024-05-21 12:26:18,55.72324,12.608316,https://urbanbelonging.app/minio/photo-uploads...,https://urbanbelonging.app/minio/photo-uploads...,4,4,"[Trees, Grass, Flowers, Other vegetation, Soil...","[Abundant, Messy, Wild]",[Peaceful],"[Construction, Birds, Wind/weather]",Fiskerihavnen,b66b51be-064e-4203-bc86-e6f81dd227c7-1600.jpeg


In [ ]:
categories = ['nature_elements', 'nature_quality', 'social_quality', 'sounds']
categories_lst = []

for cat in categories:

  # create variable and list of colname_lst
  cat_lst = str(cat)+'_lst'
  categories_lst.append(cat_lst)

  # create str from list
  df[cat] = df[cat_lst].apply(lambda x: ', '.join(x) if type(x)==list else x)

df.head(2)

,photo_id,user_id,date,datetime,lat,long,url_thumb,url_original,appeal,biodiversity_perceived,nature_elements_lst,nature_quality_lst,social_quality_lst,sounds_lst,site,filename,nature_elements,nature_quality,social_quality,sounds
0,76d4c873-3b4f-4fb9-9085-10b7104c4c1d,f48014f1-41e6-490f-9fce-b769514e2c0f,25-04-2024,2024-04-25 09:25:38,55.71664,12.609841,https://urbanbelonging.app/minio/photo-uploads...,https://urbanbelonging.app/minio/photo-uploads...,5,5,"[Grass, Trees, Flowers, Other vegetation, Bird...","[Pretty, Wild]","[Not accessible, Peaceful, Feels safe, Feeling...","[Construction, Birds, Wind/weather, Traffic]",Tunnelfabrikken,6160f900-5d6f-4cb0-8d23-39ecb1908205-1600.jpeg,"Grass, Trees, Flowers, Other vegetation, Birds...","Pretty, Wild","Not accessible, Peaceful, Feels safe, Feeling ...","Construction, Birds, Wind/weather, Traffic"
1,da370966-93f3-4bfb-9ad1-b54aace7f320,98981014-9d6c-4e37-8394-494d4f0436a6,21-05-2024,2024-05-21 12:26:18,55.72324,12.608316,https://urbanbelonging.app/minio/photo-uploads...,https://urbanbelonging.app/minio/photo-uploads...,4,4,"[Trees, Grass, Flowers, Other vegetation, Soil...","[Abundant, Messy, Wild]",[Peaceful],"[Construction, Birds, Wind/weather]",Fiskerihavnen,b66b51be-064e-4203-bc86-e6f81dd227c7-1600.jpeg,"Trees, Grass, Flowers, Other vegetation, Soil,...","Abundant, Messy, Wild",Peaceful,"Construction, Birds, Wind/weather"


In [ ]:
cat_dict = {'nature_elements_lst': ['Trees', 'Grass', 'Flowers', 'Other vegetation',
                                    'Stones/rocks', 'Soil', 'Gravel', 'Water', 'Birds',
                                    'Dogs', 'Insects', 'Other non-human species',
                                    'Wind', 'Rain', 'Sun'],
            'nature_quality_lst': ['Abundant', 'Monotonous', 'Messy', 'Ordered',
                                   'Pretty', 'Ugly', 'Protected', 'Unequally distributed',
                                   'Well cared for', 'Politically driven', 'Wild',
                                   'Conventional'],
            'social_quality_lst': ['Accessible', 'Not accessible', 'Well-protected',
                                   'Over-protected', 'Welcoming', 'Excluding', 'Peaceful',
                                   'Stressful', 'Feels safe', 'Feels unsafe', 'Feeling of belonging',
                                   'Feeling alienated'],
            'sounds_lst': ['Traffic', 'Construction', 'Birds', 'Human chatter',
                           'Water', 'Wind/weather', 'Insects']}

In [ ]:
cat_abbr = {cat: ''.join([part[0] for part in cat.split('_')[:-1]]) for cat in categories_lst}
cat_abbr

{'nature_elements_lst': 'ne',
 'nature_quality_lst': 'nq',
 'social_quality_lst': 'sq',
 'sounds_lst': 's'}

In [ ]:
processed_df = df.copy()

for i, cat in enumerate(categories_lst):

  dummy_list = cat_dict[cat]

  # one-hot encode by variable
  for var in dummy_list:
    colname = 'v' + str(i+1) + '_' + cat_abbr[cat] + '_' + var.lower().replace(' ', '_').replace('-', '_')
    processed_df[cat] = processed_df[cat].apply(lambda x: [''] if x is None or (isinstance(x, list) and len(x) == 0) else x)
    processed_df[colname] = processed_df[cat].apply(lambda x: 1 if var in x else 0)

  # add other
  other_colname = 'v' + str(i+1) + '_' + cat_abbr[cat] + '_other'
  dummy_list.append('')
  processed_df[other_colname] = processed_df[cat].apply(lambda x: 1 if x[-1] not in dummy_list else 0)

  # add other_text
  other_colname = 'v' + str(i+1) + '_' + cat_abbr[cat] + '_other_text'
  dummy_list.append('')
  processed_df[other_colname] = processed_df[cat].apply(lambda x: x[-1] if x[-1] not in dummy_list else 'No_text')

In [ ]:
processed_df = processed_df.drop(categories_lst, axis=1)

In [ ]:
processed_df.head(2)

,photo_id,user_id,date,datetime,lat,long,url_thumb,url_original,appeal,biodiversity_perceived,...,v4_s_traffic,v4_s_construction,v4_s_birds,v4_s_human_chatter,v4_s_water,v4_s_wind/weather,v4_s_insects,v4_s_,v4_s_other,v4_s_other_text
0,76d4c873-3b4f-4fb9-9085-10b7104c4c1d,f48014f1-41e6-490f-9fce-b769514e2c0f,25-04-2024,2024-04-25 09:25:38,55.71664,12.609841,https://urbanbelonging.app/minio/photo-uploads...,https://urbanbelonging.app/minio/photo-uploads...,5,5,...,1,1,1,0,0,1,0,0,0,No_text
1,da370966-93f3-4bfb-9ad1-b54aace7f320,98981014-9d6c-4e37-8394-494d4f0436a6,21-05-2024,2024-05-21 12:26:18,55.72324,12.608316,https://urbanbelonging.app/minio/photo-uploads...,https://urbanbelonging.app/minio/photo-uploads...,4,4,...,0,1,1,0,0,1,0,0,0,No_text


# 4. survey data: load to df and preprocess

## 4.1 loading df

In [ ]:
# survey_df = pd.read_excel('LIVING LAB NORDHAVN survey (svar).xlsx')

In [ ]:
survey_df.columns

In [ ]:
# rename columns
survey_df.columns = ['time', 'organization', 'name', 'email', 'participation',
                     'years_in_cph', 'zipcode', 'age', 'gender', 'sexual_orientation',
                     'racial_identity', 'physical_disability', 'mental_issue',
                     'occupation', 'agreement', 'preferred_date', 'edu_1',
                     'nordhavn_visits', 'edu_2']

In [ ]:
survey_df.head(2)

,time,organization,name,email,participation,years_in_cph,zipcode,age,gender,sexual_orientation,racial_identity,physical_disability,mental_issue,occupation,agreement,preferred_date,edu_1,nordhavn_visits,edu_2,age_group
0,2024-03-18 10:55:03.650,AAU (Tek og Etik kurset),Bastian Fitzsimmons,Bastian.fitzsimmons@gmail.com,(A) Introduction to UB App + going on photovoi...,More than 10 years,2400.0,34,Cis male/cis man,Heterosexual,Danish,"No, I have no physical disabilities",Decline to answer,Student,Utilizing the 'Urban Belonging' app involves c...,NaN,High School (Gymnasie),Never,NaN,30-39
1,2024-03-18 10:59:37.323,AAU (Tek og Etik kurset),Jochum Olsen,jochumolsen@gmail.com,(C) I am interested in participating in both A...,More than 10 years,1650.0,25,Cis male/cis man,Heterosexual,Danish,"No, I have no physical disabilities",No,Student,Utilizing the 'Urban Belonging' app involves c...,NaN,High School (Gymnasie),Never,NaN,20-29


## 4.2 preprocess data

In [ ]:
# age
survey_df.loc[survey_df['age'] == "Decline to answer", 'age'] = pd.NA
survey_df['age'] = pd.to_numeric(survey_df['age'], errors='coerce').astype('Int64')

# age_group
bins= [10,20,30,40,50,60,70,80]
labels = ['10-19','20-29','30-39','40-49','50-59','60-69','70-79']
survey_df['age_group'] = pd.cut(survey_df['age'], bins=bins, labels=labels, right=False)

# add 'Decline to answer' as a category
survey_df['age_group'] = survey_df['age_group'].cat.add_categories('Decline to answer')
survey_df['age_group'] = survey_df['age_group'].fillna('Decline to answer')

In [ ]:
# reorganise
dem_df = survey_df[['email', 'organization', 'gender', 'age', 'age_group', 'occupation',
                    'years_in_cph', 'zipcode', 'edu_1', 'sexual_orientation',
                    'racial_identity', 'physical_disability', 'mental_issue']]

In [ ]:
# zipcode
dem_df = dem_df.replace(to_replace=['Nørrebro', 'Nørrebro '], value=2200)
dem_df = dem_df.replace(to_replace='Bispebjerg', value=2400)
dem_df['zipcode'] = dem_df['zipcode'].apply(lambda x: 'Not Copenhagen' if len(str(x)) != 4 else x)

In [ ]:
# years in cph
dem_df.loc[dem_df['zipcode'] == 'Not Copenhagen', 'years_in_cph'] = "Doesn't live in Copenhagen"

In [ ]:
# emails
dem_df.loc[:, 'email'] = dem_df['email'].str.lower()

In [ ]:
# gender
gender_keep = ['Cis male/cis man', 'Cis female/cis woman', 'Genderqueer']
dem_df.loc[:, 'gender'] = dem_df['gender'].apply(lambda x: x if x in gender_keep else 'Decline to answer')
dem_df.loc[:, 'gender'] = dem_df['gender'].fillna('Decline to answer')

In [ ]:
dem_df.head(2)

In [ ]:
dem_df[dem_df['age']==19]

# 5. merge and clean

## 5.1 user id to username, email, and group

### 5.1.1 prepare username to email data

In [ ]:
username_to_email.loc[:, 'email'] = username_to_email['email'].str.lower()
username_to_email

### 5.1.2 investigate raw data['members'] (can be skipped)

In [ ]:
raw_data['members'][0]

{'id': 'e41aed45-8f15-4e5e-bef4-4875a58f23a4',
 'username': 'eva',
 'canCreatePhotoEvents': False,
 'canInviteMembers': False,
 'segments': [{'id': 'dcf06b96-c924-4da9-9c21-957bd903dd62',
   'mongoId': None,
   'createdAt': '2024-05-13T10:29:38.379Z',
   'updatedAt': '2024-05-13T10:29:38.378Z',
   'name': 'Thing Brandt Landskab',
   'groupId': '10c6b8b4-9da5-4a5e-94fb-984757d813a7'}]}

In [ ]:
# user groups
raw_data['members'][10]['segments'][0]['name']

'3XN'

### 5.1.3 create user dictionary

In [ ]:
user_ids = {}

for member in raw_data['members']:
    username = member['username']

    # Assuming username_to_email is a DataFrame or a structure where username is mapped to email
    email = username_to_email.loc[username_to_email['username'] == username, 'email'].values[0]

    # get group
    try:
      group = member['segments'][0]['name']
    except:
      group = 'no_group'

    # Store username and email in user_ids dictionary with user ID as key
    user_ids[member['id']] = {'username': username, 'user_group': group, 'email': email}

# Print the user_ids dictionary
print(user_ids)

### 5.1.4 change username of one user

In [ ]:
# find user ids
users = raw_data['members']
name1_id = next((user['id'] for user in users if user['username'] == 'name1'), None)
name2_id = next((user['id'] for user in users if user['username'] == 'name2'), None)

In [ ]:
processed_df['user_id'] = processed_df['user_id'].replace(name1_id, name2_id)

### 5.1.5 add user data to processed_df

In [ ]:
# getting username and email from user_ids dictionary
def get_username_and_email(user_id):
    if user_id in user_ids:
        return pd.Series(user_ids[user_id], index=['username', 'user_group', 'email'])
    else:
        return pd.Series([None, None], index=['username', 'user_group', 'email'])

processed_df[['username', 'user_group', 'email']] = processed_df['user_id'].apply(lambda x: get_username_and_email(x))

In [ ]:
processed_df.head(2)

## 5.2 adjust emails in dem_df

In [ ]:
# Emails in processed_df not in dem_df
emails_not_in_dem_df = processed_df[~processed_df['email'].isin(dem_df['email'])]['email']

# Emails in dem_df not in processed_df
emails_not_in_processed_df = dem_df[~dem_df['email'].isin(processed_df['email'])]['email']

print("\nEmails in processed_df not in dem_df:")
print(emails_not_in_dem_df.unique())

print("\nEmails in dem_df not in processed_df:")
print(emails_not_in_processed_df.unique())

In [ ]:
# dictionary:
correct_mails = {'wrong email': 'correct email'}

In [ ]:
dem_df.loc[:, 'email'] = dem_df['email'].replace(correct_mails)

## 5.3 merge processed_df and dem_df

In [ ]:
merged_df = pd.merge(processed_df, dem_df, on='email', how='left')
merged_df.head(2)

## 5.4 clean merged dataset

### 5.4.1 remove duplicates that were created when the datasets were merged

In [ ]:
len(processed_df)

1757

In [ ]:
processed_df[processed_df.isna().T.any()]

In [ ]:
len(merged_df)

1948

In [ ]:
len(merged_df)-len(merged_df.photo_id.unique())

191

In [ ]:
merged_df = merged_df.drop_duplicates(subset='photo_id')
len(merged_df)

1757

### 5.4.2 remove missing data
- querying emails where data is missing
- removing missing data

In [ ]:
# make list with unique emails for users that have missing data

list_emails_missingdata = merged_df[merged_df[['photo_id', 'user_id', 'date', 'lat', 'long', 'appeal',
                    'biodiversity_perceived', 'nature_elements', 'nature_quality',
                    'social_quality', 'sounds', 'nature_elements', 'nature_quality',
                    'social_quality', 'sounds']].isna().any(axis=1)]['email'].tolist()

In [ ]:
# function to get unique values
def unique(list1):
    unique_list = pd.Series(list1).drop_duplicates().tolist()
    for x in unique_list:
        print(x)

unique(list_emails_missingdata)

In [ ]:
## Get rid of missing data

cleaned_df = merged_df.dropna(subset=["url_original"]) ##drops photos without a url, aka no photo
cleaned_df = cleaned_df.dropna(subset=["appeal"]) ## drops photos without any metadata
cleaned_df = cleaned_df.dropna(subset=["lat"]) ## drops 1 observation without lat/lon
len(cleaned_df)

## we have dropped 295 rows

1460

### 5.4.3 remove rows created by Research Team

In [ ]:
cleaned_df = cleaned_df.drop(cleaned_df[cleaned_df["username"]=='researcher1'].index)
cleaned_df = cleaned_df.drop(cleaned_df[cleaned_df["username"]=='researcher2'].index)
cleaned_df = cleaned_df.drop(cleaned_df[cleaned_df["username"]=='researcher3'].index)
## we have dropped 68 rows

### 5.4.4 remove data not collected during the walks

In [ ]:
# remove data not collected during the walks
cleaned_df = cleaned_df.drop(cleaned_df[cleaned_df['site']=='tbd'].index)

### 5.4.5 fill in missing occupation data

In [ ]:
# occupation
occupation_student = ['UCPH Student', 'AAU Student', 'SLU Student']
cleaned_df.loc[cleaned_df['user_group'].isin(occupation_student), 'occupation'] = 'Student'

### 5.4.6 fill in missing data as 'Decline to answer'

In [ ]:
# age_group
cleaned_df['age_group'] = cleaned_df['age_group'].fillna('Decline to answer')

# gender
cleaned_df.loc[:, 'gender'] = cleaned_df['gender'].fillna('Decline to answer')

### 5.4.7 inspect final cleaned data

In [ ]:
len(cleaned_df)

1388

In [ ]:
cleaned_df.head(2)

# 6. export photo df

## 6.1 to .csv

In [ ]:
cleaned_df = cleaned_df.drop(['email'], axis=1)

In [ ]:
cleaned_df.head(2)

In [ ]:
import time
timestr = time.strftime("%Y.%m.%d")
print(timestr)

2024.08.08


In [ ]:
cleaned_df.to_csv(str(timestr)+' photovoice_cleaned.csv', index=False)

## 6.2 to .geojson

In [ ]:
#pip install geopandas

In [ ]:
import geopandas as gpd

In [ ]:
gdf = gpd.GeoDataFrame(
    cleaned_df, geometry=gpd.points_from_xy(cleaned_df.long, cleaned_df.lat), crs="EPSG:4326")
gdf.head(2)

In [ ]:
# categorical data to str
gdf = gdf.apply(lambda col: col.astype(str) if col.dtype.name == 'category' else col)

In [ ]:
gdf.to_file(str(timestr)+' photovoice_cleaned.geojson', driver="GeoJSON")

# 7. export participant df

In [ ]:
# check number of participants in final dataset
len(cleaned_df.user_id.unique())

97

In [ ]:
# keep the fist instance of each
cleaned_df_participants = cleaned_df.drop_duplicates(subset='user_id', keep='first')

In [ ]:
# reorganise columns
cleaned_df_participants.columns

Index(['photo_id', 'user_id', 'date', 'datetime', 'lat', 'long', 'url_thumb',
       'url_original', 'appeal', 'biodiversity_perceived', 'site', 'filename',
       'nature_elements', 'nature_quality', 'social_quality', 'sounds',
       'v1_ne_trees', 'v1_ne_grass', 'v1_ne_flowers', 'v1_ne_other_vegetation',
       'v1_ne_stones/rocks', 'v1_ne_soil', 'v1_ne_gravel', 'v1_ne_water',
       'v1_ne_birds', 'v1_ne_dogs', 'v1_ne_insects',
       'v1_ne_other_non_human_species', 'v1_ne_wind', 'v1_ne_rain',
       'v1_ne_sun', 'v1_ne_', 'v1_ne_other', 'v1_ne_other_text',
       'v2_nq_abundant', 'v2_nq_monotonous', 'v2_nq_messy', 'v2_nq_ordered',
       'v2_nq_pretty', 'v2_nq_ugly', 'v2_nq_protected',
       'v2_nq_unequally_distributed', 'v2_nq_well_cared_for',
       'v2_nq_politically_driven', 'v2_nq_wild', 'v2_nq_conventional',
       'v2_nq_', 'v2_nq_other', 'v2_nq_other_text', 'v3_sq_accessible',
       'v3_sq_not_accessible', 'v3_sq_well_protected', 'v3_sq_over_protected',
       'v3_sq_

In [ ]:
cleaned_df_participants = cleaned_df_participants[['user_id', 'username',
       'user_group', 'organization', 'gender', 'age', 'age_group',
       'occupation', 'years_in_cph', 'zipcode', 'edu_1', 'sexual_orientation',
       'racial_identity', 'physical_disability', 'mental_issue']]

In [ ]:
# export
cleaned_df_participants.to_csv(str(timestr)+' photovoice_participants_cleaned.csv', index=False)